# Fine-tuning Gemma 4 E4B for Bangla-English-Hindi code-mixing

This notebook demonstrates LoRA fine-tuning of `google/gemma-4-E4B-it` (the same model this project's API uses, see `services/api/src/app/core/config.py`) on naturally code-mixed text.

## Dataset choice

Of the datasets surveyed (SentMix-3L, EmoMix-3L, OffMix-3L, BnSentMix, L3Cube HingCorpus, HINMIX, HinGE, COMI-LINGUA), **SentMix-3L** is used here because:

- It is the only dataset that is genuinely **tri-lingual (Bangla + English + Hindi)** in naturally occurring code-mixed text — matching this project's three target languages exactly. The Hindi-English-only corpora (HingCorpus, HINMIX, HinGE, COMI-LINGUA) are missing Bengali.
- It is directly loadable from the Hugging Face Hub (`md-nishat-008/SentMix-3L`), no scraping required.

**Caveat:** SentMix-3L is published as a ~1,007-row *test* set for sentiment analysis, not a large training corpus (unlike L3Cube HingCorpus's 52.9M sentences, which is Hindi-English only and not on the Hub as raw text). That's too small to teach a 4B model new code-switching behavior from scratch, but it's enough to **demonstrate the LoRA fine-tuning mechanics end-to-end** on genuinely representative data. See the "Next steps" section at the end for how to scale this up (combining with L3Cube HingCorpus + Bengali-English sources, or LLM-generated synthetic tri-lingual conversations, per the hybrid approach recommended in the research above).

## What this notebook does

1. Loads SentMix-3L and inspects real code-mixed samples.
2. Formats each row as a sentiment-classification instruction (`text -> sentiment`), so the fine-tune target is measurable, not just vibes.
3. Loads Gemma 4 E4B in 4-bit and attaches a LoRA adapter (fits comfortably on a single 32GB GPU, e.g. the RTX PRO 4500 used for this project's inference pod).
4. Trains a few epochs over the small dataset.
5. Compares base vs. fine-tuned model output on held-out code-mixed examples.

In [ ]:
# Run once per environment. transformers/torch/accelerate already come from the
# project's `gemma` extra (services/api/pyproject.toml); peft/trl/bitsandbytes are
# only needed for this fine-tuning notebook.
%pip install -q "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.44" "datasets>=3.0"

In [ ]:
import os

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

MODEL_ID = "google/gemma-4-E4B-it"  # same model_id as services/api/src/app/core/config.py
DATASET_ID = "md-nishat-008/SentMix-3L"
OUTPUT_DIR = "gemma4-e4b-codemix-lora"
HF_TOKEN = os.environ.get("HF_TOKEN")  # gated model; requires accepted access on HF Hub

## 1. Load and inspect SentMix-3L

Print a few raw rows first, to confirm the text is genuinely code-mixed across all three languages before we train on it.

In [ ]:
raw_dataset = load_dataset(DATASET_ID)
print(raw_dataset)

# The dataset ships as a single split; carve out a small held-out set for
# before/after comparison later.
split_dataset = raw_dataset["test"].train_test_split(test_size=0.1, seed=42)
train_rows, eval_rows = split_dataset["train"], split_dataset["test"]
print(f"train rows: {len(train_rows)}, eval rows: {len(eval_rows)}")

for row in train_rows.select(range(3)):
    print(row)

## 2. Format as instruction-tuning examples

Each row becomes a short instruction: given the code-mixed sentence, predict its sentiment label. This keeps the fine-tuning objective concrete (measurable accuracy) while the underlying text the model attends to and learns from is the natural tri-lingual code-switching itself.

In [ ]:
PROMPT_TEMPLATE = (
    "Classify the sentiment of this Bangla-English-Hindi code-mixed text as "
    "positive, negative, or neutral.\n\nText: {text}\nSentiment:"
)


def format_example(row: dict) -> dict:
    return {
        "prompt": PROMPT_TEMPLATE.format(text=row["text"]),
        "completion": f" {row['label']}",
    }


formatted_train = train_rows.map(format_example, remove_columns=train_rows.column_names)
formatted_eval = eval_rows.map(format_example, remove_columns=eval_rows.column_names)
print(formatted_train[0])

## 3. Load Gemma 4 E4B in 4-bit and attach LoRA

`target_modules="all-linear"` lets PEFT find every linear projection inside the wrapped multimodal architecture without having to hardcode Gemma's internal module names.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN,
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 4. Train

A handful of epochs is enough on ~900 rows to demonstrate the mechanics; this is a fine-tuning *demo*, not a production training run (see the dataset-size caveat above).

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    dataset_text_field=None,
    formatting_func=lambda row: row["prompt"] + row["completion"],
    processing_class=processor.tokenizer,
    max_seq_length=256,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)

## 5. Before / after comparison

Run the same held-out code-mixed prompts through the base model and the fine-tuned adapter.

In [ ]:
def generate(prompt: str, *, use_adapter: bool) -> str:
    if use_adapter:
        model.set_adapter("default")
        active_model = model
    else:
        active_model = model.get_base_model()

    inputs = processor.tokenizer(prompt, return_tensors="pt").to(active_model.device)
    with torch.no_grad():
        generated = active_model.generate(**inputs, max_new_tokens=8, do_sample=False)
    return processor.tokenizer.decode(
        generated[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True
    )


for row in formatted_eval.select(range(5)):
    print("Prompt:", row["prompt"])
    print("Expected:", row["completion"].strip())
    print("Base model:", generate(row["prompt"], use_adapter=False).strip())
    print("Fine-tuned:", generate(row["prompt"], use_adapter=True).strip())
    print("-" * 60)

## Next steps for a production-scale fine-tune

SentMix-3L (~1k rows) is enough to prove the pipeline works, not enough to substantially move a 4B model's code-mixing fluency. To scale up, per the research this notebook is based on:

1. Combine SentMix-3L / EmoMix-3L / OffMix-3L (tri-lingual, but small and label-focused) with L3Cube HingCorpus (52.9M Hindi-English sentences, Hindi-English only, available at `github.com/l3cube-pune/code-mixed-nlp`) for volume.
2. Add a Bengali-English source (e.g. BnSentMix, 20k rows) so Bengali isn't underrepresented relative to Hindi.
3. Since no large natural tri-lingual (Bangla+Hindi+English) conversational corpus currently exists publicly, generate synthetic tri-lingual conversational data with an LLM and have bilingual/trilingual reviewers spot-check a sample — the hybrid approach is the most practical path today.
4. Switch the training objective from single-label classification (used here for a measurable demo) to plain next-token / instruction-following on the raw conversational text, since the actual goal is fluent code-mixed generation, not classification.